Необхідно виконати завдання, використовуючи  pandas dataframe, проаналізувати часові витрати на виконання процедур (профілювання часу виконання здійснити за допомогою модуля timeit).

## Завдання 1:
Звантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset 


In [1]:
import pandas as pd
import numpy as np
import os
import urllib.request
import zipfile
import timeit

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_file = "household_power_consumption.txt"

if not os.path.exists(txt_file):
    print("Файл не знайдено. Початок завантаження архіву...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Архів завантажено та розпаковано успішно.")
else:
    print("Файл вже існує.")

start_time = timeit.default_timer()
df = pd.read_csv(txt_file, sep=';', na_values=['?'], low_memory=False)
end_time = timeit.default_timer()

print(f"Початкова кількість рядків: {len(df)}")
print(f"Час зчитування файлу: {end_time - start_time:.4f} сек.")

df.head(10)

Файл вже існує.
Початкова кількість рядків: 2075259
Час зчитування файлу: 2.4477 сек.


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
5,16/12/2006,17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0
6,16/12/2006,17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0
7,16/12/2006,17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0
8,16/12/2006,17:32:00,3.668,0.510,233.99,15.8,0.0,1.0,17.0
9,16/12/2006,17:33:00,3.662,0.510,233.86,15.8,0.0,2.0,16.0


## Завдання 2: 
Здійснити data cleaning

In [2]:
numeric_columns = [
    'Global_active_power', 'Global_reactive_power', 'Voltage', 
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

missing_before = df.isna().sum().sum()
print(f"Знайдено порожніх клітинок (NaN) до очищення: {missing_before}")

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

start_time = timeit.default_timer()

df_cleaned = df.dropna()

end_time = timeit.default_timer()

print(f"Залишилося рядків: {len(df_cleaned)}")
print(f"Видалено рядки: {len(df) - len(df_cleaned)}")
print(f"Час виконання очищення: {end_time - start_time:.4f} сек.")

print("\nТипи даних після конвертації:")
print(df_cleaned[numeric_columns].dtypes)

df = df_cleaned

Знайдено порожніх клітинок (NaN) до очищення: 181853
Залишилося рядків: 2049280
Видалено рядки: 25979
Час виконання очищення: 0.4328 сек.

Типи даних після конвертації:
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object


## Завдання 3:
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [3]:
def filter_high_power(dataframe):
    return dataframe[dataframe['Global_active_power'] > 5.0]

start_time = timeit.default_timer()

res_task3 = filter_high_power(df)

execution_time = timeit.default_timer() - start_time
print(f"Знайдено записів: {len(res_task3)}")
print(f"Час виконання: {execution_time:.4f}")
res_task3.head()

Знайдено записів: 17547
Час виконання: 0.0081


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


## Завдання 4: 
Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.


In [4]:
def filter_intensity_appliances(dataframe):
    return dataframe[(dataframe['Global_intensity'] >= 19.0) & 
                     (dataframe['Global_intensity'] <= 20.0) & 
                     (dataframe['Sub_metering_2'] > dataframe['Sub_metering_3'])]

start_time = timeit.default_timer()

res_task4 = filter_intensity_appliances(df)

execution_time = timeit.default_timer() - start_time
print(f"Знайдено записів: {len(res_task4)}")
print(f"Час виконання: {execution_time:.4f}")
res_task4.head()

Знайдено записів: 2509
Час виконання: 0.0246


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


## Завдання 5: 
Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [5]:
def get_random_sample_means(dataframe):
    sample = dataframe.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

start_time = timeit.default_timer()

res_task5 = get_random_sample_means(df)

execution_time = timeit.default_timer() - start_time
print(f"Час виконання: {execution_time:.4f}")
print("Середні значення (Вт-год):")
print(res_task5)

Час виконання: 0.3066
Середні значення (Вт-год):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


## Завдання 6: 
Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [6]:
def filter_evening_and_slice(dataframe):
    evening_data = dataframe[(dataframe['Time'] >= '18:00:00') & 
                             (dataframe['Global_active_power'] > 6.0) & 
                             (dataframe['Sub_metering_2'] > dataframe['Sub_metering_1']) & 
                             (dataframe['Sub_metering_2'] > dataframe['Sub_metering_3'])]

    half_index = len(evening_data) // 2
    first_half = evening_data.iloc[:half_index]
    second_half = evening_data.iloc[half_index:]

    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]

    return pd.concat([res_first, res_second])

start_time = timeit.default_timer()

res_task6 = filter_evening_and_slice(df)

execution_time = timeit.default_timer() - start_time
print(f"Фінальна кількість записів після зрізів: {len(res_task6)}")
print(f"Час виконання: {execution_time:.4f}")
res_task6.head(3)

Фінальна кількість записів після зрізів: 310
Час виконання: 0.2359


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0


## Завдання 7: 
Пронормувати та стандартизувати вибраний датасет
Нормалізація: 

In [15]:
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
data_num = df[numeric_cols].copy()

start_norm = timeit.default_timer()

normalized_df = (data_num - data_num.min()) / (data_num.max() - data_num.min())

end_norm = timeit.default_timer()

print(f"Час виконання нормалізації: {end_norm - start_norm:.6f} сек.")
print("Дані після нормалізації:")
display(normalized_df.head())

Час виконання нормалізації: 0.382626 сек.
Дані після нормалізації (діапазон 0-1):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


Стандартизація:

In [17]:
start_std = timeit.default_timer()

# Формула: (X - mean) / std
standardized_df = (data_num - data_num.mean()) / data_num.std()

end_std = timeit.default_timer()

print(f"Час виконання стандартизації: {end_std - start_std:.6f} сек.")
print("Дані після стандартизації:")
display(standardized_df.head())

Час виконання стандартизації: 0.473784 сек.
Дані після стандартизації:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955076,2.610720,-1.851816,3.098788,-0.182337,-0.051274,1.249420
1,4.037084,2.770405,-2.225274,4.133799,-0.182337,-0.051274,1.130897
2,4.050325,3.320431,-2.330213,4.133799,-0.182337,0.120487,1.249420
3,4.063566,3.355916,-2.191323,4.133799,-0.182337,-0.051274,1.249420
4,2.434881,3.586572,-1.592555,2.513781,-0.182337,-0.051274,1.249420


## Завдання 8: 
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.
1. Коефіцієнт Пірсона:

In [19]:
start_p = timeit.default_timer()

pearson_val = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')

print(f"Пірсон: {pearson_val:.6f}")
print(f"Час: {timeit.default_timer() - start_p:.4f} сек.")

Пірсон: 0.998889
Час: 0.0487 сек.


2. Коефіцієнт Спірмена:

In [18]:
start_s = timeit.default_timer()

spearman_val = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Спірмен: {spearman_val:.6f}")
print(f"Час: {timeit.default_timer() - start_s:.4f} сек.")

Спірмен: 0.995372
Час: 0.6697 сек.


## Завдання 9: 
Провести One Hot Encoding категоріального атрибута.

In [14]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Month'] = df['Date'].dt.month

start_ohe = timeit.default_timer()

df_sample = df.sample(50000, random_state=42)
df_encoded = pd.get_dummies(df_sample, columns=['Month'], prefix='month')

execution_ohe = timeit.default_timer() - start_ohe

print(f"Час виконання One Hot Encoding: {execution_ohe:.6f} сек.")
print(f"Кількість рядків в обробці: {len(df_encoded)}")
print(f"Нові колонки: {[col for col in df_encoded.columns if 'month_' in col]}")

display(df_encoded.head())

Час виконання One Hot Encoding: 0.113068 сек.
Кількість рядків в обробці: 50000
Нові колонки: ['month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,month_1,...,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
1030580,2008-12-01,09:44:00,1.502,0.074,240.17,6.4,0.0,0.0,18.0,False,...,False,False,False,False,False,False,False,False,False,True
1815,2006-12-17,23:39:00,0.374,0.264,245.50,1.8,0.0,2.0,0.0,False,...,False,False,False,False,False,False,False,False,False,True
1295977,2009-06-03,17:01:00,0.620,0.300,239.85,3.0,0.0,1.0,1.0,False,...,False,False,False,True,False,False,False,False,False,False
206669,2007-05-09,05:53:00,0.280,0.200,235.72,1.4,0.0,0.0,0.0,False,...,False,False,True,False,False,False,False,False,False,False
1048893,2008-12-14,02:57:00,1.372,0.054,243.95,5.6,0.0,0.0,18.0,False,...,False,False,False,False,False,False,False,False,False,True
